# Лабораторная работа №4
## Сравнительный анализ алгоритмов классификации

**Цель работы:** изучить методы классификации, выполнить подбор гиперпараметров и сравнить качество моделей машинного обучения.

Используемые алгоритмы:
- kNN
- Decision Tree
- SVM
- MLP Neural Network


In [54]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

import kagglehub
import os

import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings(
    "ignore",
    category=ConvergenceWarning
)

sns.set_theme(style="whitegrid")


## Датасет №1 — Pima Indians Diabetes

**Бизнес-задача:** определить вероятность наличия диабета у пациента.

**Объект:** пациент.

**Признаки:** медицинские показатели пациента.

**Целевая переменная:** Outcome:
- 0 — диабет отсутствует
- 1 — диабет присутствует


In [55]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

columns = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

df1 = pd.read_csv(url, names=columns)

df1.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [ ]:
df1.info()
df1.describe()


In [ ]:
X = df1.drop("Outcome", axis=1)
y = df1["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


## Подбор моделей с GridSearchCV

In [ ]:
models = {
    "kNN": (
        KNeighborsClassifier(),
        {
            "n_neighbors": [3, 5, 7, 9]
        }
    ),

    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {
            "max_depth": [3, 5, 10, None]
        }
    ),

    "SVM": (
        SVC(),
        {
            "C": [0.1, 1, 10],
            "kernel": ["linear", "rbf"]
        }
    ),

    "MLP": (
        MLPClassifier(
            max_iter=1500,
            random_state=42,
            early_stopping=True
        ),
        {
            "hidden_layer_sizes": [
                (50,),
                (100,),
                (50, 50)
            ],
            "activation": ["relu", "tanh"]
        }
    )
}

results1 = []
best_models1 = {}

feature_names = df1.drop("Outcome", axis=1).columns

for name, (model, param) in models.items():

    print("=" * 70)
    print(f"АЛГОРИТМ: {name}")
    print("=" * 70)

    grid = GridSearchCV(
        model,
        param,
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    pred = best_model.predict(X_test)

    best_models1[name] = best_model

    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)

    results1.append([
        name,
        grid.best_params_,
        accuracy,
        precision,
        recall,
        f1
    ])

    print("Лучшие параметры:")
    print(grid.best_params_)

    print("\nClassification Report:")
    print(classification_report(y_test, pred))

    # График для kNN
    if name == "kNN":

        cv_results = pd.DataFrame(grid.cv_results_)

        k_values = cv_results["param_n_neighbors"].astype(int)
        cv_scores = cv_results["mean_test_score"]

        plt.figure(figsize=(8, 5))

        plt.plot(
            k_values,
            cv_scores,
            marker="o",
            linewidth=2
        )

        plt.xlabel("Количество соседей k")
        plt.ylabel("Accuracy")
        plt.title("kNN — зависимость Accuracy от k")
        plt.xticks(k_values)
        plt.grid(True)
        plt.show()

    # График для Decision Tree
    elif name == "Decision Tree":

        importance = pd.Series(
            best_model.feature_importances_,
            index=feature_names
        ).sort_values(ascending=True)

        plt.figure(figsize=(8, 5))

        importance.plot(kind="barh")

        plt.xlabel("Важность")
        plt.ylabel("Признак")
        plt.title("Decision Tree — важность признаков")
        plt.tight_layout()
        plt.show()

    # График для SVM
    elif name == "SVM":

        scores = best_model.decision_function(X_test)

        fpr, tpr, thresholds = roc_curve(y_test, scores)
        roc_auc = auc(fpr, tpr)

        plt.figure(figsize=(8, 5))

        plt.plot(
            fpr,
            tpr,
            linewidth=2,
            label=f"AUC = {roc_auc:.3f}"
        )

        plt.plot(
            [0, 1],
            [0, 1],
            linestyle="--"
        )

        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("SVM — ROC-кривая")
        plt.legend()
        plt.grid(True)
        plt.show()

    # График для MLP
    elif name == "MLP":

        plt.figure(figsize=(8, 5))

        plt.plot(
            best_model.loss_curve_,
            linewidth=2
        )

        plt.xlabel("Итерация")
        plt.ylabel("Loss")
        plt.title("MLP — изменение функции потерь")
        plt.grid(True)
        plt.show()

    # Матрица ошибок для всех моделей
    cm = confusion_matrix(y_test, pred)

    plt.figure(figsize=(5, 4))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues"
    )

    plt.title(f"{name} — матрица ошибок")
    plt.xlabel("Предсказанный класс")
    plt.ylabel("Истинный класс")
    plt.show()

In [ ]:
table1 = pd.DataFrame(
    results1,
    columns=[
        "Model",
        "Best parameters",
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ]
)


In [ ]:
table1=pd.DataFrame(
    results1,
    columns=[
        "Model",
        "Best parameters",
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ]
)

table1


# Датасет №2 — Bank Customer Churn Prediction

**Бизнес-задача:** определить, уйдет ли клиент из банка.

**Целевая переменная:** Exited.


In [ ]:
path2 = kagglehub.dataset_download(
    "shrutimechlearn/churn-modelling"
)

csv_file=[f for f in os.listdir(path2) if f.endswith(".csv")][0]

df2=pd.read_csv(os.path.join(path2,csv_file))

df2.head()


In [ ]:
df2.info()


In [ ]:
df2=df2.drop(
    ["RowNumber","CustomerId","Surname"],
    axis=1
)

df2=pd.get_dummies(
    df2,
    drop_first=True
)

X=df2.drop("Exited",axis=1)
y=df2["Exited"]

X_train,X_test,y_train,y_test=train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler=StandardScaler()

X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)


In [ ]:
results2=[]

for name,(model,param) in models.items():
    grid=GridSearchCV(model,param,cv=5,scoring="accuracy")
    grid.fit(X_train,y_train)

    pred=grid.predict(X_test)

    results2.append([
        name,
        grid.best_params_,
        accuracy_score(y_test,pred),
        precision_score(y_test,pred),
        recall_score(y_test,pred),
        f1_score(y_test,pred)
    ])

    print(name)
    print(classification_report(y_test,pred))

    sns.heatmap(confusion_matrix(y_test,pred),
                annot=True,
                fmt="d")
    plt.title(name)
    plt.show()


In [ ]:
table2=pd.DataFrame(
    results2,
    columns=[
        "Model",
        "Best parameters",
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ]
)

table2


# Вывод

В работе были исследованы четыре алгоритма классификации.
Для каждого алгоритма выполнен подбор гиперпараметров с помощью GridSearchCV.

Качество моделей оценивалось по:
- Accuracy
- Precision
- Recall
- F1-score

Полученные результаты позволяют сравнить эффективность различных методов классификации на двух наборах данных.